# Wannier-centre shifts between base and displaced runs

Formerly `Y2Ir2O7/phonon/wannier_center_rms.ipynb`; set `MATERIAL`. It walks `data/<MATERIAL>/` for `trial_*`
runs itself (any `trial_*` folder with a `_centres.xyz`).

# Wannier center RMS difference: base vs phonon-mode calculations

Runs are discovered by searching for `*_centres.xyz` anywhere under a
`trial_NN_<projections>` directory. The depth below the trial directory varies —
files may sit directly in it, or under a variant subdir (`proj_gauge`, `dis_ml`,
`dis_no_ml`, `rigid_shift`, `no_displacement`), optionally with a jobid below
that — so nothing here assumes a fixed path depth. The older
`<...>/output/<jobid>` layout is ignored.

**Chart 1 — difference vs base.** Compares Wannier centers index-to-index using
the minimum-image convention in the base lattice. Each mode run is paired only
with base runs carrying the **same trial name**: the trial fixes the projection
set, hence the number and ordering of the Wannier functions, so a cross-trial
comparison is meaningless (and the counts do not even match — trial_01 → 224,
trial_02 → 176, trial_03 → 216, trial_04 → 248). Where a trial has several base
variants, each pair is listed separately, since which base gauge a mode run
should match is exactly what is under test. This chart is only physically
meaningful when both runs converged to the same Wannier gauge with the same WF
ordering — a large RMS for a 0.01 Å displacement signals a gauge/ordering
mismatch, not real center motion.

**Chart 2 — distance to atoms, within each structure.** For each run separately,
the RMS distance of every Wannier center to its **nearest atom in that same
structure**, folded in that run's **own lattice**. Atoms are the non-`X` entries
of the same `*_centres.xyz`. This pairing is **gauge/ordering-independent**, so
it needs no base run at all: every discovered run gets a bar, including trials
that have no mode counterpart and are therefore absent from chart 1.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Root to search: the material folder in data/ (base/ and phonon/<irrep>/ below it).
ROOT = DATA / "Y2Ir2O7"

# Only runs living under a `trial_NN_<projections>` directory are considered;
# the older `<...>/output/<jobid>` layout is ignored.  The depth *below* the
# trial directory varies (files may sit directly in it, or under a variant
# subdir such as proj_gauge/ or dis_ml/, optionally with a jobid below that),
# so runs are discovered by looking for the centres file rather than by a
# fixed-depth glob.
TRIAL_RE = re.compile(r"^trial_\d+")
JOBID_RE = re.compile(r"^\d{5,}[a-z0-9]*$")

In [ ]:
def discover_runs(root=ROOT):
    """Every finished Wannier run under a `trial_*` directory.

    A "run" is any directory containing a `*_centres.xyz`.  Metadata is read off
    the path by matching each component against a pattern rather than by
    position, so this survives further changes in directory depth.  Runs not
    under a `trial_*` component (the legacy `output/<jobid>` layout) are skipped.
    """
    runs = []
    for centres_file in sorted(root.glob("**/*_centres.xyz")):
        parts = centres_file.parent.relative_to(root).parts
        trial_at = next((i for i, p in enumerate(parts) if TRIAL_RE.match(p)), None)
        if trial_at is None:
            continue
        below = parts[trial_at + 1:]
        runs.append(dict(
            path=centres_file.parent,
            centres_file=centres_file,
            trial=parts[trial_at],
            # everything below the trial dir, minus a trailing jobid
            variant="/".join(p for p in below if not JOBID_RE.match(p)) or "-",
            jobid=next((p for p in below if JOBID_RE.match(p)), ""),
            is_base=parts[0] == "base",
            irrep=parts[1] if parts[0] == "phonon" else parts[0],
            mode=next((p for p in parts if p.startswith("mode")), ""),
            amp=next((p for p in parts if p.startswith("Q_")), ""),
            U=next((p[2:] for p in parts if p.startswith("u_")), ""),
        ))
    return runs


def read_centres(path):
    """Read Wannier centers (the 'X' entries) from a *_centres.xyz file, in Angstrom."""
    centres = []
    with open(path) as f:
        f.readline()  # entry count
        f.readline()  # comment
        for line in f:
            parts = line.split()
            if parts and parts[0] == "X":
                centres.append([float(x) for x in parts[1:4]])
    return np.array(centres)


def read_atoms(path):
    """Read nominal atomic positions (the non-'X' entries) from a *_centres.xyz file, in Angstrom."""
    atoms = []
    with open(path) as f:
        f.readline()  # entry count
        f.readline()  # comment
        for line in f:
            parts = line.split()
            if parts and parts[0] != "X":
                atoms.append([float(x) for x in parts[1:4]])
    return np.array(atoms)


def read_lattice(out_dir):
    """Read lattice vectors (rows, in Angstrom) from the .wout file in out_dir."""
    wouts = [p for p in sorted(Path(out_dir).glob("*.wout")) if not p.stem.endswith("_pp")]
    lat = []
    with open(wouts[0]) as f:
        for line in f:
            if "Lattice Vectors" in line:
                for _ in range(3):
                    lat.append([float(x) for x in f.readline().split()[1:4]])
                break
    return np.array(lat)


def min_image_diff(centres, centres_ref, lat):
    """Cartesian center differences folded to the nearest periodic image."""
    d_frac = (centres - centres_ref) @ np.linalg.inv(lat)
    d_frac -= np.round(d_frac)
    return d_frac @ lat


def nearest_atom_dist(centres, atoms, lat):
    """For each Wannier center, the min-image Cartesian distance to its nearest atom.

    Unlike the index-to-index base difference, this pairing is independent of the
    Wannier gauge / ordering, so it is a robust measure of how far each WF sits
    from a nominal atomic site (and, for a mode run, whether centers track the
    displaced atoms).
    """
    d = centres[:, None, :] - atoms[None, :, :]        # (n_wann, n_atom, 3)
    d_frac = d @ np.linalg.inv(lat)
    d_frac -= np.round(d_frac)
    d_cart = d_frac @ lat
    return np.linalg.norm(d_cart, axis=2).min(axis=1)  # (n_wann,)


runs = discover_runs()
inventory = pd.DataFrame([
    dict(kind="base" if r["is_base"] else r["irrep"], mode=r["mode"], trial=r["trial"],
         variant=r["variant"], jobid=r["jobid"], n_wann=len(read_centres(r["centres_file"])))
    for r in runs
])
print(f"{len(runs)} runs under trial_* directories")
inventory

In [ ]:
# Chart 1 data: mode run vs base run, paired by TRIAL.
#
# The trial directory names the projection set, so it fixes the number and the
# ordering of the Wannier functions.  Comparing across trials is meaningless
# (and the counts would not even match: trial_01 -> 224, trial_02 -> 176,
# trial_03 -> 216, trial_04 -> 248), so every mode run is compared only against
# base runs carrying the same trial name.  Where a trial has several base
# variants (proj_gauge, dis_no_ml, ...) each pair is listed separately, since
# which base gauge a mode run should be compared to is exactly the thing under
# test.
base_runs = [r for r in runs if r["is_base"]]
mode_runs = [r for r in runs if not r["is_base"]]

rows, unpaired = [], []
for m in mode_runs:
    partners = [b for b in base_runs if b["trial"] == m["trial"]]
    if not partners:
        unpaired.append(m)
        continue
    centres = read_centres(m["centres_file"])
    for b in partners:
        base_centres = read_centres(b["centres_file"])
        if len(centres) != len(base_centres):
            print(f"count mismatch within {m['trial']}: "
                  f"{m['path']} has {len(centres)}, {b['path']} has {len(base_centres)}")
            continue
        lat = read_lattice(b["path"])  # compare in the base lattice
        norms = np.linalg.norm(min_image_diff(centres, base_centres, lat), axis=1)
        rows.append(dict(
            irrep=m["irrep"], mode=m["mode"], amp=m["amp"], U=m["U"],
            trial=m["trial"],
            mode_variant=m["variant"], mode_jobid=m["jobid"],
            base_variant=b["variant"], base_jobid=b["jobid"],
            n_wann=len(centres),
            rms=np.sqrt(np.mean(norms**2)),
            max=np.max(norms),
        ))

for m in unpaired:
    print(f"no base run for trial {m['trial']}: {m['path']}")

df = (pd.DataFrame(rows)
      .sort_values(["trial", "irrep", "mode", "amp", "U", "mode_jobid", "base_jobid"])
      .reset_index(drop=True))
df

In [ ]:
trial_colors = dict(zip(sorted(df["trial"].unique()), plt.cm.tab10.colors))
labels = (df["mode"] + " " + df["mode_variant"]
          + "  vs  base " + df["base_variant"]
          + "\n" + df["trial"])
y = np.arange(len(df))

fig, ax = plt.subplots(figsize=(9, 0.55 * len(df) + 1.2))
ax.barh(y, df["rms"], color=[trial_colors[t] for t in df["trial"]])
for yi, v in zip(y, df["rms"]):
    ax.text(v + 0.01, yi, f"{v:.3f}", va="center", fontsize=8)

ax.set_yticks(y, labels, fontsize=8)
ax.invert_yaxis()
ax.margins(x=0.18)
ax.grid(axis="x", alpha=0.3)
ax.set_xlabel(r"RMS $\Delta\bar{\mathbf{r}}$ ($\mathrm{\AA}$)")
ax.set_title("Wannier center RMS difference, mode vs base (paired by trial)")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2 data: Wannier center distance to the nearest atom WITHIN EACH STRUCTURE.
# Base WF centers vs base atoms, mode WF centers vs that mode's atoms, each
# folded in its OWN lattice.  Gauge/ordering-independent, so no pairing is
# needed and every discovered run gets a bar -- including trials that have no
# mode counterpart and so are absent from chart 1.
atom_rows = []
for r in runs:
    centres = read_centres(r["centres_file"])
    lat_run = read_lattice(r["path"])  # this structure's own lattice
    dist = nearest_atom_dist(centres, read_atoms(r["centres_file"]), lat_run)
    kind = "base" if r["is_base"] else f"{r['irrep']} {r['mode']}"
    atom_rows.append(dict(
        label=f"{kind} {r['variant']}\n{r['trial']}",
        kind=kind, trial=r["trial"], variant=r["variant"], jobid=r["jobid"],
        n_wann=len(centres),
        rms_atom=np.sqrt(np.mean(dist**2)),
        max_atom=float(np.max(dist)),
    ))

df_atom = (pd.DataFrame(atom_rows)
           .sort_values(["trial", "kind", "variant", "jobid"])
           .reset_index(drop=True))
df_atom

In [ ]:
# Same visual as chart 1, but every run gets a bar and each uses its own atoms
# and own lattice.  Colored by trial, since the projection set is what sets the
# expected scale here.
trial_colors_atom = dict(zip(sorted(df_atom["trial"].unique()), plt.cm.tab10.colors))
ya = np.arange(len(df_atom))

fig, ax = plt.subplots(figsize=(9, 0.55 * len(df_atom) + 1.2))
ax.barh(ya, df_atom["rms_atom"], color=[trial_colors_atom[t] for t in df_atom["trial"]])
for yi, v in zip(ya, df_atom["rms_atom"]):
    ax.text(v + 0.01, yi, f"{v:.3f}", va="center", fontsize=8)

ax.set_yticks(ya, df_atom["label"], fontsize=8)
ax.invert_yaxis()
ax.margins(x=0.18)
ax.grid(axis="x", alpha=0.3)
ax.set_xlabel(r"RMS $\Delta\bar{\mathbf{r}}$ ($\mathrm{\AA}$)")
ax.set_title("Wannier center RMS distance to nearest atom (each structure)")
plt.tight_layout()
plt.show()